# Lesson 2.1 — Chart File Structure

**Phase 2: Chart Anatomy & Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Create a new chart with `helm create`
- Understand the purpose of every file and directory in a chart
- Distinguish between `application` and `library` chart types

---

## Creating a New Chart

The fastest way to get started is `helm create`, which scaffolds a complete chart:

In [ ]:
# Create a new chart called 'myapp' in /tmp
cd /tmp && helm create myapp

# Show everything that was created
find /tmp/myapp -type f | sort

---

## The Standard Chart Structure

```
myapp/
├── Chart.yaml           # Chart identity: name, version, appVersion, type
├── values.yaml          # Default configuration values
├── values.schema.json   # (optional) JSON Schema to validate values
├── charts/              # Bundled sub-charts (dependencies)
├── crds/                # Custom Resource Definitions (applied before templates)
├── templates/
│   ├── NOTES.txt        # Post-install message (templated)
│   ├── _helpers.tpl     # Named template snippets (not rendered directly)
│   ├── deployment.yaml  # Kubernetes Deployment resource
│   ├── service.yaml     # Kubernetes Service resource
│   ├── serviceaccount.yaml
│   ├── ingress.yaml
│   ├── hpa.yaml         # HorizontalPodAutoscaler
│   └── tests/
│       └── test-connection.yaml   # Helm test hook
└── .helmignore          # Files to exclude when packaging
```

> **Reference:** [Charts Guide](https://helm.sh/docs/topics/charts/) — https://helm.sh/docs/topics/charts/

---

## Key Files Explained

In [ ]:
# 1. Chart.yaml — the identity card
cat /tmp/myapp/Chart.yaml

In [ ]:
# 2. values.yaml — default configuration (top section)
head -30 /tmp/myapp/values.yaml

In [ ]:
# 3. _helpers.tpl — named template definitions (starts with _ so not rendered)
cat /tmp/myapp/templates/_helpers.tpl

In [ ]:
# 4. NOTES.txt — shown to user after helm install
cat /tmp/myapp/templates/NOTES.txt

---

## The `.helmignore` File

Works like `.gitignore` — patterns listed here are excluded when `helm package` creates the chart archive.

In [ ]:
cat /tmp/myapp/.helmignore

---

## Application vs Library Charts

The `type` field in `Chart.yaml` controls whether your chart:

| Type | Purpose | Installable? |
|---|---|---|
| `application` | Deploys actual Kubernetes resources | Yes |
| `library` | Provides reusable template helpers for other charts | No |

Library charts are imported as dependencies and their named templates (`define` blocks in `_helpers.tpl`) can be called from the parent chart. Their resource templates are NOT rendered.

---

## Build a Minimal Chart From Scratch

In [ ]:
# Create a minimal chart manually (no scaffolding)
rm -rf /tmp/minimalapp
mkdir -p /tmp/minimalapp/templates

# Write Chart.yaml
cat > /tmp/minimalapp/Chart.yaml << 'EOF'
apiVersion: v2
name: minimalapp
description: A hand-crafted minimal chart
type: application
version: 0.1.0
appVersion: "1.0.0"
EOF

# Write a minimal values.yaml
cat > /tmp/minimalapp/values.yaml << 'EOF'
replicaCount: 1
image:
  repository: nginx
  tag: "stable"
EOF

# Write a simple template
cat > /tmp/minimalapp/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-config
data:
  app: {{ .Chart.Name }}
  version: {{ .Chart.AppVersion }}
EOF

echo "Minimal chart created:"
find /tmp/minimalapp -type f | sort

In [ ]:
# Render it with helm template to see the output
helm template my-release /tmp/minimalapp

---

## Linting Your Chart

In [ ]:
# helm lint checks for common errors in chart structure and templates
helm lint /tmp/minimalapp
echo "---"
helm lint /tmp/myapp

---

## Exercise

In [ ]:
# TASK 1: Create a new chart called 'webapp' in /tmp
# TODO:


# TASK 2: Open /tmp/webapp/Chart.yaml and change the description
#         to 'My first custom Helm chart'
# TODO:


# TASK 3: Run helm lint on your chart and fix any issues
# TODO:


# TASK 4: Run helm template on your chart and count how many
#         Kubernetes resources it would create
# TODO:
# helm template my-webapp /tmp/webapp | grep -c '^kind:'

---

## Summary

- `helm create <name>` scaffolds a fully-featured chart with best-practice defaults.
- The key files are: `Chart.yaml` (identity), `values.yaml` (defaults), `templates/` (K8s YAML with placeholders).
- Files starting with `_` in `templates/` are **not** rendered as K8s resources — they hold reusable helpers.
- `helm lint` catches structural and template errors early.
- `helm template` renders to plain YAML without a cluster.

---

## Next Lesson

➡️ [Lesson 2.2 — Chart.yaml & values.yaml](lesson-2.2-chart-yaml-and-values.ipynb)